# Project 6: SBERT within-project temporal story-point estimation

Design (assigned configuration: **SBERT sentence embeddings, within-project, temporal ordering**):

- SBERT `all-MiniLM-L6-v2` sentence embeddings (384 dimensions), computed in the embedding notebook and aligned to issues by ID.
- A separate model for each software project in TAWOS.
- Chronological ~80:20 development/test split per project (test = the latest ~20% of issues).
- Five rolling-origin validation folds inside the development data (details below).
- Hyperparameters and epoch count selected independently per project from development data only.
- MAE and MdAE on the original story-point scale; a median-predictor baseline and an optional ridge-regression sanity check.

**Required files (one folder, set as `DATA_DIR` in Cell 1):** `issue_clean.csv`, `project.csv`, `splits_temporal.json`, `sbert_ids.csv`, `sbert_minilm_emb.npy`.

**Validation protocol note.** The development data are cut into 6 chronological blocks. Fold *k* trains on blocks 0..k and validates on block k+1, so every validation block comes after its training data. Block 0 is only ever training data. This is a chronological interpretation of the guideline's five-fold requirement; it differs from shuffled K-fold. Confirm the interpretation with the instructor.

**Early stopping.** Early stopping monitors MAE on the most recent 15% of the *training window* (an inner holdout); the validation fold is scored once, after training. If a training window is too small for an inner holdout (fewer than about 64 issues), the model trains for a fixed `max_epochs` with no early stopping.

**Known limitations (also state them in the report).**
1. The first folds train on a small part of the development data, so they are noisy.
2. CV scores of the selected configuration are slightly optimistic (best of several configurations on the same folds); test scores are not affected.
3. Exact duplicate texts can occur in both development and test data; Cell 7 reports how many and the metrics without them.
4. The CNN slides over embedding dimensions, which have no spatial order; the ridge check in Cell 9 shows whether the CNN adds anything.
5. Results come from one training run per configuration (plus a 3-seed sensitivity check for the final models); GPU runs are not guaranteed to be bit-exact.

In [4]:
# ============================================================
# CELL 1: Setup. Paths only (no personal Drive IDs, no API login)
# ============================================================
import os, sys, json, math, random, time, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/TAWOS_Project6_Files"
    RESULTS_DIR = os.environ.get("TAWOS_RESULTS_DIR", "/content/drive/MyDrive/project6_results")
else:
    DATA_DIR    = os.environ.get("TAWOS_DATA_DIR",    "./data")
    RESULTS_DIR = os.environ.get("TAWOS_RESULTS_DIR", "./results_project6")

INPUT_FILES = ["issue_clean.csv", "project.csv", "splits_temporal.json",
               "sbert_ids.csv", "sbert_minilm_emb.npy"]
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
missing = [f for f in INPUT_FILES if not os.path.isfile(os.path.join(DATA_DIR, f))]
assert not missing, f"Missing input files in {DATA_DIR}: {missing}"
print("DATA_DIR   :", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR   : /content/drive/MyDrive/TAWOS_Project6_Files
RESULTS_DIR: /content/drive/MyDrive/project6_results


In [5]:
# ============================================================
# CELL 2: Configuration (saved to config.json for the repository)
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SMOKE = os.environ.get("P6_SMOKE") == "1"   # tiny run to test the code only; never use for reported results

CONFIG = {
    "assignment": "Project 6: SBERT sentence embeddings, within-project, temporal ordering",
    "sbert_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dim": 384,
    "seed": 42,
    "final_seeds": [42, 43, 44],           # seed sensitivity for the FINAL models only
    "n_folds": 5,
    "dev_test_split": "chronological ~80:20 per project",
    "validation": "rolling-origin: development data in 6 chronological blocks; "
                  "fold k trains on blocks 0..k and validates on block k+1",
    "batch_size": 64, "max_epochs": 60, "patience": 8,
    "weight_decay": 1e-4, "hidden": 64, "grad_clip": 1.0,
    "inner_holdout_fraction": 0.15, "min_inner_rows": 10,
    "hp_grid": [
        {"lr": 1e-3, "dropout": 0.2}, {"lr": 1e-3, "dropout": 0.4},
        {"lr": 5e-4, "dropout": 0.2}, {"lr": 5e-4, "dropout": 0.4},
    ],
    "run_ridge_check": True,
    "ridge_alphas": [0.1, 1.0, 10.0, 100.0, 1000.0],
    "sp_max": 100.0,
    "loss": "HuberLoss(delta=1.0)", "optimizer": "Adam",
    "target_transform": "log1p, then standardised on the rows used for fitting; inverse transform and clip to [0, sp_max]",
    "smoke_test": SMOKE,
}
if SMOKE:
    CONFIG.update(max_epochs=3, patience=2, final_seeds=[42, 43],
                  hp_grid=CONFIG["hp_grid"][:2], ridge_alphas=[1.0, 100.0])

SEED, N_FOLDS = CONFIG["seed"], CONFIG["n_folds"]
BATCH_SIZE, MAX_EPOCHS, PATIENCE = CONFIG["batch_size"], CONFIG["max_epochs"], CONFIG["patience"]
WEIGHT_DECAY, HIDDEN, GRAD_CLIP = CONFIG["weight_decay"], CONFIG["hidden"], CONFIG["grad_clip"]
INNER_FRAC, MIN_INNER = CONFIG["inner_holdout_fraction"], CONFIG["min_inner_rows"]
SP_MAX, HP_GRID = CONFIG["sp_max"], CONFIG["hp_grid"]
FINAL_SEEDS = CONFIG["final_seeds"]
CFG_HASH = hashlib.md5(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:8]

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True      # reduces, but does not guarantee, run-to-run differences
torch.backends.cudnn.benchmark = False
print("Device:", device, "| config hash:", CFG_HASH, "| SMOKE:", SMOKE)
if device == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
with open(f"{RESULTS_DIR}/config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)
print("Saved", f"{RESULTS_DIR}/config.json")

Device: cpu | config hash: 2f4c77d2 | SMOKE: False
Saved /content/drive/MyDrive/project6_results/config.json


In [6]:
# ============================================================
# CELL 3: Load data, align embeddings and labels, validate splits
# ============================================================
CLEAN_PATH = os.path.join(DATA_DIR, "issue_clean.csv")
PROJ_PATH  = os.path.join(DATA_DIR, "project.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "splits_temporal.json")
EMB_PATH   = os.path.join(DATA_DIR, "sbert_minilm_emb.npy")
IDS_PATH   = os.path.join(DATA_DIR, "sbert_ids.csv")

df = pd.read_csv(CLEAN_PATH, parse_dates=["Creation_Date"], low_memory=False)
required = {"ID", "Project_ID", "Story_Point", "Creation_Date"}
assert not (required - set(df.columns)), f"issue_clean.csv is missing: {sorted(required - set(df.columns))}"
df["ID"] = pd.to_numeric(df["ID"], errors="raise").astype("int64")
df["Project_ID"] = pd.to_numeric(df["Project_ID"], errors="raise").astype("int64")
df["Story_Point"] = pd.to_numeric(df["Story_Point"], errors="coerce").astype("float64")
assert df["ID"].is_unique, "Issue IDs must be unique."
assert df["Creation_Date"].notna().all(), "Creation_Date has missing/invalid values."
assert np.isfinite(df["Story_Point"].to_numpy()).all(), "Story_Point has missing or non-finite values."
assert (df["Story_Point"] >= 0).all(), "Negative story points (log1p needs >= 0)."
assert (df["Story_Point"] <= SP_MAX).all(), f"Labels above SP_MAX={SP_MAX}; decide explicitly how to handle them."

proj_df = pd.read_csv(PROJ_PATH)
assert {"ID", "Project_Key", "Name"} <= set(proj_df.columns), "project.csv needs ID, Project_Key, Name."
proj_df["ID"] = pd.to_numeric(proj_df["ID"], errors="raise").astype("int64")
assert proj_df["ID"].is_unique
proj_key_map  = dict(zip(proj_df["ID"], proj_df["Project_Key"].astype(str)))
proj_name_map = dict(zip(proj_df["ID"], proj_df["Name"].astype(str)))
dataset_project_ids = set(df["Project_ID"].unique())
assert dataset_project_ids <= set(proj_df["ID"]), "Some projects are missing from project.csv."

emb_raw = np.load(EMB_PATH, mmap_mode="r")
assert emb_raw.ndim == 2 and emb_raw.shape[1] == CONFIG["embedding_dim"], f"Unexpected embedding shape {emb_raw.shape}"
ids_emb = pd.read_csv(IDS_PATH)
ids_emb["ID"] = pd.to_numeric(ids_emb["ID"], errors="raise").astype("int64")
assert len(emb_raw) == len(ids_emb) and ids_emb["ID"].is_unique, "Embedding rows and ID list do not match."
emb_pos = pd.Series(np.arange(len(ids_emb)), index=ids_emb["ID"].values)
assert not (set(df["ID"]) - set(emb_pos.index)), "Some cleaned issues have no SBERT embedding."
emb = np.asarray(emb_raw[[int(emb_pos.loc[i]) for i in df["ID"]]], dtype="float32")
assert np.isfinite(emb).all(), "Embeddings contain NaN/inf."

id_to_emb_idx = pd.Series(np.arange(len(df)), index=df["ID"].values)
id_to_y    = pd.Series(df["Story_Point"].values, index=df["ID"].values)
id_to_proj = pd.Series(df["Project_ID"].values, index=df["ID"].values)
id_to_date = pd.Series(df["Creation_Date"].values, index=df["ID"].values)
id_to_dnum = pd.Series(df["Creation_Date"].astype("int64").values, index=df["ID"].values)   # fast sorting key
id_to_text = (pd.Series(df["text"].astype(str).str.strip().str.lower().values, index=df["ID"].values)
              if "text" in df.columns else None)
print(f"{len(df)} issues, {len(dataset_project_ids)} projects, embeddings {emb.shape}",
      "| text column available" if id_to_text is not None else "| no text column (duplicate check skipped)")

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = {int(k): v for k, v in json.load(f).items()}
assert set(splits) == dataset_project_ids, "Projects in splits_temporal.json and issue_clean.csv differ."

for pid, split in sorted(splits.items()):
    dev_ids  = [int(x) for x in split["dev"]]
    test_ids = [int(x) for x in split["test"]]
    assert dev_ids and test_ids, f"Project {pid}: empty dev/test."
    assert len(set(dev_ids)) == len(dev_ids) and len(set(test_ids)) == len(test_ids), f"Project {pid}: duplicate IDs."
    assert set(dev_ids).isdisjoint(test_ids), f"Project {pid}: dev/test overlap."
    combined = set(dev_ids) | set(test_ids)
    eligible = set(df.loc[df["Project_ID"] == pid, "ID"].astype(int))
    assert combined == eligible, f"Project {pid}: split does not cover exactly the cleaned issues."
    assert id_to_dnum.loc[dev_ids].max() <= id_to_dnum.loc[test_ids].min(), f"Project {pid}: test is not later than dev."
    n_total = len(dev_ids) + len(test_ids)
    assert abs(len(test_ids) - round(0.20 * n_total)) <= 1, f"Project {pid}: test size is not ~20%."

    folds = split.get("folds", [])
    assert len(folds) == N_FOLDS, f"Project {pid}: expected {N_FOLDS} folds, found {len(folds)}."
    val_blocks, train_blocks = [], []
    for fi, fold in enumerate(folds):
        tr, va = [int(x) for x in fold["train"]], [int(x) for x in fold["val"]]
        assert tr and va and set(tr).isdisjoint(va), f"Project {pid}, fold {fi}: empty or overlapping."
        assert set(tr) <= set(dev_ids) and set(va) <= set(dev_ids), f"Project {pid}, fold {fi}: non-dev issue."
        assert id_to_dnum.loc[tr].max() <= id_to_dnum.loc[va].min(), f"Project {pid}, fold {fi}: future issue in training."
        val_blocks.append(set(va)); train_blocks.append(set(tr))
    for fi in range(1, N_FOLDS):
        assert id_to_dnum.loc[list(val_blocks[fi-1])].max() <= id_to_dnum.loc[list(val_blocks[fi])].min(), \
            f"Project {pid}: validation blocks not chronological at fold {fi}."
        assert val_blocks[fi-1].isdisjoint(val_blocks[fi]) and val_blocks[fi-1] <= train_blocks[fi] \
            and train_blocks[fi-1] <= train_blocks[fi], f"Project {pid}: folds are not expanding/nested."
    warmup, val_union = train_blocks[0], set().union(*val_blocks)
    assert warmup.isdisjoint(val_union) and (warmup | val_union) == set(dev_ids), \
        f"Project {pid}: folds do not partition development data into warm-up + five validation blocks."
    assert all(train_blocks[i] == (warmup | set().union(*val_blocks[:i])) for i in range(N_FOLDS)), \
        f"Project {pid}: training windows are not warm-up plus all earlier validation blocks."
print(f"[PASS] all {len(splits)} projects: dataset/embedding alignment, 80:20 temporal split, rolling-origin folds.")

65015 issues, 39 projects, embeddings (65015, 384) | text column available
[PASS] all 39 projects: dataset/embedding alignment, 80:20 temporal split, rolling-origin folds.


In [7]:
# ============================================================
# CELL 4: Dataset, model, training utilities
#   Early stopping uses an INNER chronological holdout taken from the
#   training window. The validation fold is scored once, after training.
# ============================================================
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sort_by_date(ids):
    arr = np.asarray(ids, dtype="int64")
    return arr[np.argsort(id_to_dnum.loc[arr].values, kind="stable")].tolist()

class IssueDataset(Dataset):
    def __init__(self, ids, X, y):
        self.ids, self.X, self.y = np.asarray(ids, dtype="int64"), X, y
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.X[i], self.y[i]

class SBERT_CNN(nn.Module):
    """Three-layer 1D CNN over the 384-d SBERT sentence vector."""
    def __init__(self, in_dim=384, hidden=64, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn1, self.bn2, self.bn3 = nn.GroupNorm(8, 32), nn.GroupNorm(8, 64), nn.GroupNorm(8, 128)
        self.pool, self.gpool = nn.MaxPool1d(2), nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc1, self.fc2 = nn.Linear(128, hidden), nn.Linear(hidden, 1)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.dropout(self.gpool(x).squeeze(-1))
        return self.fc2(F.relu(self.fc1(x))).squeeze(-1)

def train_one_epoch(model, loader, opt, crit, device):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        crit(model(xb), yb).backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()

@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    out = [model(xb.to(device)).cpu().numpy() for xb, _ in loader]
    return np.concatenate(out) if out else np.array([])

def inverse_target(z, mu, sigma):
    pred = np.expm1(np.asarray(z, dtype="float64") * sigma + mu)
    pred = np.nan_to_num(pred, nan=0.0, posinf=SP_MAX, neginf=0.0)
    return np.clip(pred, 0.0, SP_MAX)

def compute_metrics(y_true, y_pred):
    ae = np.abs(np.asarray(y_true, dtype="float64") - np.clip(np.asarray(y_pred, dtype="float64"), 0.0, None))
    return float(np.mean(ae)), float(np.median(ae))

def eff_batch_size(n_rows, bs):
    if n_rows < 1:
        raise ValueError("Empty training set.")
    return int(min(bs, n_rows, max(1, int(np.ceil(n_rows / 4)))))

def fit_one_run(train_ids, val_ids, hp, device, seed=SEED):
    """Fit on an earlier window; use only an inner chronological tail for early stopping.
    If the window is too small for an inner holdout, train for MAX_EPOCHS (no early stopping)."""
    set_seed(seed)
    train_ids, val_ids = sort_by_date(train_ids), sort_by_date(val_ids)
    n_tr = len(train_ids)
    n_inner = int(round(n_tr * INNER_FRAC))
    use_inner = n_inner >= MIN_INNER and (n_tr - n_inner) >= MIN_INNER
    fit_ids, es_ids = (train_ids[:-n_inner], train_ids[-n_inner:]) if use_inner else (train_ids, [])

    X_tr, X_va = emb[id_to_emb_idx.loc[fit_ids].values], emb[id_to_emb_idx.loc[val_ids].values]
    y_tr = id_to_y.loc[fit_ids].values.astype("float64")
    y_va = id_to_y.loc[val_ids].values.astype("float64")
    y_tr_log = np.log1p(y_tr)
    mu, sd = float(y_tr_log.mean()), float(y_tr_log.std())
    if not np.isfinite(sd) or sd < 1e-8:
        sd = 1.0
    y_tr_n = ((y_tr_log - mu) / sd).astype("float32")

    bs = eff_batch_size(len(fit_ids), BATCH_SIZE)
    tr_loader = DataLoader(IssueDataset(fit_ids, X_tr, y_tr_n), batch_size=bs, shuffle=True, drop_last=False)
    va_loader = DataLoader(IssueDataset(val_ids, X_va, np.zeros(len(val_ids), dtype="float32")),
                           batch_size=max(bs * 2, 16), shuffle=False)
    if use_inner:
        y_es = id_to_y.loc[es_ids].values.astype("float64")
        es_loader = DataLoader(IssueDataset(es_ids, emb[id_to_emb_idx.loc[es_ids].values],
                                            np.zeros(len(es_ids), dtype="float32")),
                               batch_size=max(bs * 2, 16), shuffle=False)

    model = SBERT_CNN(in_dim=X_tr.shape[1], hidden=HIDDEN, dropout=hp["dropout"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=WEIGHT_DECAY)
    crit = nn.HuberLoss(delta=1.0)
    best_es, best_state, best_epoch, patience, epochs_run = float("inf"), None, MAX_EPOCHS, 0, MAX_EPOCHS

    for epoch in range(1, MAX_EPOCHS + 1):
        train_one_epoch(model, tr_loader, opt, crit, device)
        if use_inner:
            es_mae, _ = compute_metrics(y_es, inverse_target(predict(model, es_loader, device), mu, sd))
            if es_mae < best_es - 1e-6:
                best_es, best_epoch, patience = es_mae, epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= PATIENCE:
                    epochs_run = epoch
                    break
    if use_inner and best_state is not None:
        model.load_state_dict(best_state)
    val_pred = inverse_target(predict(model, va_loader, device), mu, sd)
    val_mae, val_mdae = compute_metrics(y_va, val_pred)
    return {"val_mae": val_mae, "val_mdae": val_mdae, "best_epoch": int(best_epoch),
            "es_source": "inner_chronological_holdout" if use_inner else "fixed_max_epochs_small_training_window",
            "n_fit": len(fit_ids), "n_es": len(es_ids), "batch_size_eff": bs, "epochs_run": int(epochs_run)}

In [8]:
# ============================================================
# CELL 5: Five-fold temporal CV for every configuration and project
#   Progress is checkpointed after each configuration (safe against Colab disconnects).
# ============================================================
PARTIAL = f"{RESULTS_DIR}/cv_results_partial.csv"
RESUME = True
project_ids_all = sorted(splits.keys())

def run_cv_for_config(hp, name):
    rows = []
    for k, pid in enumerate(project_ids_all, 1):
        for fold_idx, fold in enumerate(splits[pid]["folds"]):
            r = fit_one_run(fold["train"], fold["val"], hp, device)
            rows.append({"cfg_hash": CFG_HASH, "config": name, "project_id": pid, "fold": fold_idx,
                         "n_train": len(fold["train"]), "n_val": len(fold["val"]), **r})
        if k % 10 == 0 or k == len(project_ids_all):
            print(f"[{name}] {k}/{len(project_ids_all)} projects done")
    return pd.DataFrame(rows)

frames, done = [], set()
if RESUME and os.path.isfile(PARTIAL):
    prev = pd.read_csv(PARTIAL)
    prev = prev[prev["cfg_hash"] == CFG_HASH]
    if len(prev):
        frames.append(prev); done = set(prev["config"].unique())
        print("Resuming; configurations already finished:", sorted(done))

t0 = time.time()
for i, hp in enumerate(HP_GRID):
    name = f"cfg{i}_lr{hp['lr']}_do{hp['dropout']}"
    if name in done:
        continue
    print(f"\n=== CV for {name} ===")
    frames.append(run_cv_for_config(hp, name))
    pd.concat(frames, ignore_index=True).to_csv(PARTIAL, index=False)

cv_df = pd.concat(frames, ignore_index=True)
cv_df.to_csv(f"{RESULTS_DIR}/cv_results_all_configs.csv", index=False)
assert len(cv_df) == len(project_ids_all) * N_FOLDS * len(HP_GRID), "Incomplete CV grid."
assert cv_df.groupby(["project_id", "config"]).size().eq(N_FOLDS).all(), "Each project/config needs five folds."
print(f"\nCV finished ({(time.time()-t0)/60:.1f} min this session); rows={len(cv_df)}")
print("Folds using inner early stopping:", int((cv_df['es_source'] == 'inner_chronological_holdout').sum()),
      "of", len(cv_df))


=== CV for cfg0_lr0.001_do0.2 ===
[cfg0_lr0.001_do0.2] 10/39 projects done
[cfg0_lr0.001_do0.2] 20/39 projects done
[cfg0_lr0.001_do0.2] 30/39 projects done
[cfg0_lr0.001_do0.2] 39/39 projects done

=== CV for cfg1_lr0.001_do0.4 ===
[cfg1_lr0.001_do0.4] 10/39 projects done
[cfg1_lr0.001_do0.4] 20/39 projects done
[cfg1_lr0.001_do0.4] 30/39 projects done
[cfg1_lr0.001_do0.4] 39/39 projects done

=== CV for cfg2_lr0.0005_do0.2 ===
[cfg2_lr0.0005_do0.2] 10/39 projects done
[cfg2_lr0.0005_do0.2] 20/39 projects done
[cfg2_lr0.0005_do0.2] 30/39 projects done
[cfg2_lr0.0005_do0.2] 39/39 projects done

=== CV for cfg3_lr0.0005_do0.4 ===
[cfg3_lr0.0005_do0.4] 10/39 projects done
[cfg3_lr0.0005_do0.4] 20/39 projects done
[cfg3_lr0.0005_do0.4] 30/39 projects done
[cfg3_lr0.0005_do0.4] 39/39 projects done

CV finished (182.7 min this session); rows=780
Folds using inner early stopping: 696 of 780


In [9]:
# ============================================================
# CELL 6: Select hyperparameters and epoch count independently per project
#   Epoch rule: median best epoch over the folds that used an inner early-stopping
#   holdout. Folds trained for a fixed MAX_EPOCHS carry no information about the
#   best epoch and are excluded. If no fold had an inner holdout, use MAX_EPOCHS.
# ============================================================
config_lookup = {f"cfg{i}_lr{hp['lr']}_do{hp['dropout']}": hp for i, hp in enumerate(HP_GRID)}
rows, selected = [], []
for pid, group in cv_df.groupby("project_id"):
    scores = (group.groupby("config")
                   .agg(mean_val_mae=("val_mae", "mean"), mean_val_mdae=("val_mdae", "mean"),
                        n_folds=("fold", "nunique")).reset_index())
    scores = scores[scores["n_folds"] == N_FOLDS].sort_values(["mean_val_mae", "config"])
    assert not scores.empty, f"Project {pid}: no complete configuration."
    name = str(scores.iloc[0]["config"])
    chosen_rows = group[group["config"] == name]
    es_rows = chosen_rows[chosen_rows["es_source"] == "inner_chronological_holdout"]
    if len(es_rows):
        epochs, rule = int(max(1, round(es_rows["best_epoch"].median()))), "median best epoch of folds with inner early stopping"
    else:
        epochs, rule = MAX_EPOCHS, "fixed MAX_EPOCHS (no fold had an inner holdout)"
    rows.append({"project_id": int(pid), "best_config_name": name,
                 "best_lr": config_lookup[name]["lr"], "best_dropout": config_lookup[name]["dropout"],
                 "cv_mean_mae": float(chosen_rows["val_mae"].mean()), "cv_std_mae": float(chosen_rows["val_mae"].std()),
                 "cv_mean_mdae": float(chosen_rows["val_mdae"].mean()),
                 "final_epochs": epochs, "epoch_rule": rule, "n_folds_with_inner_es": int(len(es_rows))})
    selected.append(chosen_rows.copy())

project_selection_df = pd.DataFrame(rows).sort_values("project_id")
best_cv = pd.concat(selected, ignore_index=True)
best_config_by_project = {int(r["project_id"]): config_lookup[r["best_config_name"]] for r in rows}
project_epochs = {int(r["project_id"]): int(r["final_epochs"]) for r in rows}

project_selection_df.to_csv(f"{RESULTS_DIR}/best_config_per_project.csv", index=False)
best_cv.to_csv(f"{RESULTS_DIR}/cv_results_selected_per_project.csv", index=False)
print(project_selection_df.to_string(index=False))

 project_id    best_config_name  best_lr  best_dropout  cv_mean_mae  cv_std_mae  cv_mean_mdae  final_epochs                                           epoch_rule  n_folds_with_inner_es
          1  cfg1_lr0.001_do0.4   0.0010           0.4     2.231549    0.386827      1.477541             3 median best epoch of folds with inner early stopping                      5
          2 cfg3_lr0.0005_do0.4   0.0005           0.4     3.474241    1.624717      1.919107             2 median best epoch of folds with inner early stopping                      3
          3  cfg0_lr0.001_do0.2   0.0010           0.2     0.931397    0.231998      0.717578             9 median best epoch of folds with inner early stopping                      5
          4  cfg1_lr0.001_do0.4   0.0010           0.4     1.421951    0.100885      0.726575             4 median best epoch of folds with inner early stopping                      5
          5 cfg3_lr0.0005_do0.4   0.0005           0.4     0.961960    0.051171 

In [10]:
# ============================================================
# CELL 7: Final held-out test evaluation (test set used once per final model)
#   * trained on the full development set with the per-project epoch count
#   * primary result = seed 42; seeds in FINAL_SEEDS give a sensitivity check
#   * trivial baselines (median / mean of development labels)
#   * duplicate-text diagnostic: test issues whose exact text also occurs in development data
# ============================================================
def train_final_and_predict(pid, seed):
    s = splits[pid]
    dev_ids, test_ids = sort_by_date(s["dev"]), sort_by_date(s["test"])
    set_seed(seed)
    X_tr, X_te = emb[id_to_emb_idx.loc[dev_ids].values], emb[id_to_emb_idx.loc[test_ids].values]
    y_tr = id_to_y.loc[dev_ids].values.astype("float64")
    y_tr_log = np.log1p(y_tr)
    mu, sd = float(y_tr_log.mean()), float(y_tr_log.std())
    if not np.isfinite(sd) or sd < 1e-8:
        sd = 1.0
    y_tr_n = ((y_tr_log - mu) / sd).astype("float32")
    bs = eff_batch_size(len(dev_ids), BATCH_SIZE)
    tr_loader = DataLoader(IssueDataset(dev_ids, X_tr, y_tr_n), batch_size=bs, shuffle=True, drop_last=False)
    te_loader = DataLoader(IssueDataset(test_ids, X_te, np.zeros(len(test_ids), dtype="float32")),
                           batch_size=bs * 2, shuffle=False)
    hp = best_config_by_project[pid]
    model = SBERT_CNN(in_dim=emb.shape[1], hidden=HIDDEN, dropout=hp["dropout"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=WEIGHT_DECAY)
    crit = nn.HuberLoss(delta=1.0)
    for _ in range(project_epochs[pid]):
        train_one_epoch(model, tr_loader, opt, crit, device)
    return dev_ids, test_ids, inverse_target(predict(model, te_loader, device), mu, sd)

per_project_final, seed_rows, test_pred_records = [], [], []
for pid in project_ids_all:
    for seed in FINAL_SEEDS:
        dev_ids, test_ids, te_pred = train_final_and_predict(pid, seed)
        y_te = id_to_y.loc[test_ids].values.astype("float64")
        mae, mdae = compute_metrics(y_te, te_pred)
        seed_rows.append({"project_id": pid, "seed": seed, "test_mae": mae, "test_mdae": mdae})
        if seed == SEED:
            y_dev = id_to_y.loc[dev_ids].values.astype("float64")
            b_med_mae, b_med_mdae = compute_metrics(y_te, np.full_like(y_te, float(np.median(y_dev))))
            b_mean_mae, b_mean_mdae = compute_metrics(y_te, np.full_like(y_te, float(np.mean(y_dev))))
            n_dup, mae_x, mdae_x = np.nan, np.nan, np.nan
            if id_to_text is not None:
                dev_texts = set(id_to_text.loc[dev_ids])
                dup = np.array([id_to_text.loc[i] in dev_texts for i in test_ids])
                n_dup = int(dup.sum())
                if n_dup < len(test_ids):
                    mae_x, mdae_x = compute_metrics(y_te[~dup], te_pred[~dup])
            per_project_final.append({
                "project_id": pid, "project_key": proj_key_map[pid], "project_name": proj_name_map[pid],
                "n_dev": len(dev_ids), "n_test": len(test_ids), "epochs": project_epochs[pid],
                "selected_lr": best_config_by_project[pid]["lr"], "selected_dropout": best_config_by_project[pid]["dropout"],
                "test_mae": mae, "test_mdae": mdae,
                "baseline_median_mae": b_med_mae, "baseline_median_mdae": b_med_mdae,
                "baseline_mean_mae": b_mean_mae, "baseline_mean_mdae": b_mean_mdae,
                "n_test_exact_dup_of_dev": n_dup, "test_mae_excl_dup": mae_x, "test_mdae_excl_dup": mdae_x})
            test_pred_records += [{"project_id": pid, "project_key": proj_key_map[pid], "issue_id": int(i),
                                   "y_true": float(t), "y_pred": float(p)} for i, t, p in zip(test_ids, y_te, te_pred)]
    r = per_project_final[-1]
    print(f"{r['project_key']:<10s} dev={r['n_dev']:>5} test={r['n_test']:>5} epochs={r['epochs']:>2} "
          f"MAE={r['test_mae']:.3f} MdAE={r['test_mdae']:.3f} (median-baseline MAE={r['baseline_median_mae']:.3f})")

final_df      = pd.DataFrame(per_project_final).sort_values("project_id")
seed_df       = pd.DataFrame(seed_rows)
test_preds_df = pd.DataFrame(test_pred_records)
final_df.to_csv(f"{RESULTS_DIR}/test_metrics_per_project.csv", index=False)
seed_df.to_csv(f"{RESULTS_DIR}/test_metrics_by_seed.csv", index=False)
test_preds_df.to_csv(f"{RESULTS_DIR}/test_predictions.csv", index=False)

print("\nSeed sensitivity (macro mean over projects):")
print(seed_df.groupby("seed")[["test_mae", "test_mdae"]].mean().round(4).to_string())
print("Projects beating the median baseline (MAE):",
      int((final_df["test_mae"] < final_df["baseline_median_mae"]).sum()), "/", len(final_df))
if id_to_text is not None:
    print("Test issues with an exact-duplicate text in development data:",
          int(final_df["n_test_exact_dup_of_dev"].sum()), "of", int(final_df["n_test"].sum()))

XD         dev= 2956 test=  739 epochs= 3 MAE=1.904 MdAE=1.995 (median-baseline MAE=1.904)
DATACASS   dev=  184 test=   47 epochs= 2 MAE=2.370 MdAE=1.801 (median-baseline MAE=2.426)
NEXUS      dev= 1475 test=  369 epochs= 9 MAE=1.158 MdAE=0.736 (median-baseline MAE=1.206)
MESOS      dev= 2609 test=  653 epochs= 4 MAE=1.556 MdAE=0.691 (median-baseline MAE=1.479)
USERGRID   dev=  380 test=   95 epochs= 4 MAE=1.260 MdAE=0.621 (median-baseline MAE=1.168)
MXNET      dev=  158 test=   40 epochs= 1 MAE=4.535 MdAE=1.592 (median-baseline MAE=3.900)
ALOY       dev=  252 test=   63 epochs= 4 MAE=2.312 MdAE=1.893 (median-baseline MAE=1.730)
APSTUD     dev=  711 test=  178 epochs= 3 MAE=3.651 MdAE=1.845 (median-baseline MAE=3.983)
CLI        dev=  299 test=   75 epochs= 2 MAE=2.153 MdAE=2.394 (median-baseline MAE=1.947)
DAEMON     dev=  192 test=   49 epochs= 4 MAE=5.657 MdAE=2.724 (median-baseline MAE=5.673)
TIDOC      dev= 1036 test=  260 epochs= 3 MAE=3.527 MdAE=1.614 (median-baseline MAE=3.373)

In [11]:
# ============================================================
# CELL 8: Summaries. Macro mean over projects is the MAIN number; pooled is shown beside it.
# ============================================================
cv_per_project = (best_cv.groupby("project_id")
                  .agg(cv_mae_mean=("val_mae", "mean"), cv_mae_sd=("val_mae", "std"),
                       cv_mdae_mean=("val_mdae", "mean"), cv_mdae_sd=("val_mdae", "std"),
                       n_folds=("fold", "nunique")).reset_index())
assert (cv_per_project["n_folds"] == N_FOLDS).all()
cv_per_project.to_csv(f"{RESULTS_DIR}/cv_per_project_summary.csv", index=False)
cv_per_fold = (best_cv.groupby("fold")
               .agg(macro_mae=("val_mae", "mean"), across_project_sd_mae=("val_mae", "std"),
                    macro_mdae=("val_mdae", "mean"), across_project_sd_mdae=("val_mdae", "std"),
                    n_projects=("project_id", "nunique"), mean_n_train=("n_train", "mean")).reset_index())
cv_per_fold.to_csv(f"{RESULTS_DIR}/cv_per_fold_macro_summary.csv", index=False)

pooled_mae, pooled_mdae = compute_metrics(test_preds_df["y_true"].values, test_preds_df["y_pred"].values)
seed_macro = seed_df.groupby("seed")[["test_mae", "test_mdae"]].mean()
summary = pd.DataFrame([
    {"split": "5-fold rolling-origin CV", "metric": "MAE",  "main_macro_mean": cv_per_project["cv_mae_mean"].mean(),
     "sd_across_projects": cv_per_project["cv_mae_mean"].std(ddof=1), "mean_within_project_fold_sd": cv_per_project["cv_mae_sd"].mean(), "pooled": np.nan},
    {"split": "5-fold rolling-origin CV", "metric": "MdAE", "main_macro_mean": cv_per_project["cv_mdae_mean"].mean(),
     "sd_across_projects": cv_per_project["cv_mdae_mean"].std(ddof=1), "mean_within_project_fold_sd": cv_per_project["cv_mdae_sd"].mean(), "pooled": np.nan},
    {"split": "held-out temporal test", "metric": "MAE",  "main_macro_mean": final_df["test_mae"].mean(),
     "sd_across_projects": final_df["test_mae"].std(ddof=1), "mean_within_project_fold_sd": np.nan, "pooled": pooled_mae},
    {"split": "held-out temporal test", "metric": "MdAE", "main_macro_mean": final_df["test_mdae"].mean(),
     "sd_across_projects": final_df["test_mdae"].std(ddof=1), "mean_within_project_fold_sd": np.nan, "pooled": pooled_mdae},
    {"split": "held-out test: median baseline", "metric": "MAE",  "main_macro_mean": final_df["baseline_median_mae"].mean(),
     "sd_across_projects": final_df["baseline_median_mae"].std(ddof=1), "mean_within_project_fold_sd": np.nan, "pooled": np.nan},
    {"split": "held-out test: median baseline", "metric": "MdAE", "main_macro_mean": final_df["baseline_median_mdae"].mean(),
     "sd_across_projects": final_df["baseline_median_mdae"].std(ddof=1), "mean_within_project_fold_sd": np.nan, "pooled": np.nan},
])
summary.insert(0, "representation", "SBERT all-MiniLM-L6-v2 + three-layer 1D CNN")
summary.to_csv(f"{RESULTS_DIR}/summary_report.csv", index=False)
print(summary.round(4).to_string(index=False))
print(f"\nSeed sensitivity of the macro test MAE: min={seed_macro['test_mae'].min():.4f}, max={seed_macro['test_mae'].max():.4f}")

run_meta = {**CONFIG, "config_hash": CFG_HASH, "device": device, "n_projects": len(project_ids_all),
            "project_epochs": {str(k): v for k, v in project_epochs.items()},
            "best_config_by_project": {str(k): v for k, v in best_config_by_project.items()},
            "epoch_rule": "median best epoch of folds with inner early stopping; MAX_EPOCHS if none",
            "baselines": "project-specific median and mean of development labels"}
with open(f"{RESULTS_DIR}/run_meta.json", "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)
print("Saved summary_report.csv, run_meta.json")

                             representation                          split metric  main_macro_mean  sd_across_projects  mean_within_project_fold_sd  pooled
SBERT all-MiniLM-L6-v2 + three-layer 1D CNN       5-fold rolling-origin CV    MAE           2.2756              1.9447                       0.7067     NaN
SBERT all-MiniLM-L6-v2 + three-layer 1D CNN       5-fold rolling-origin CV   MdAE           1.5057              1.1137                       0.4621     NaN
SBERT all-MiniLM-L6-v2 + three-layer 1D CNN         held-out temporal test    MAE           1.9772              1.3321                          NaN  2.7048
SBERT all-MiniLM-L6-v2 + three-layer 1D CNN         held-out temporal test   MdAE           1.3260              1.0333                          NaN  1.3515
SBERT all-MiniLM-L6-v2 + three-layer 1D CNN held-out test: median baseline    MAE           1.9441              1.3622                          NaN     NaN
SBERT all-MiniLM-L6-v2 + three-layer 1D CNN held-out test: media

In [12]:
# ============================================================
# CELL 9 (optional): Ridge-regression sanity check, same protocol
#   Same rolling folds, same target transform, alpha chosen on development folds only.
#   Shows whether the CNN adds anything over a simple linear model on the same embeddings.
# ============================================================
ridge_df = None
if CONFIG["run_ridge_check"]:
    from sklearn.linear_model import Ridge
    ALPHAS = CONFIG["ridge_alphas"]

    def ridge_fit_predict(fit_ids, pred_ids, alpha):
        fit_ids, pred_ids = sort_by_date(fit_ids), sort_by_date(pred_ids)
        y_log = np.log1p(id_to_y.loc[fit_ids].values.astype("float64"))
        mu, sd = float(y_log.mean()), float(y_log.std())
        if not np.isfinite(sd) or sd < 1e-8:
            sd = 1.0
        m = Ridge(alpha=alpha).fit(emb[id_to_emb_idx.loc[fit_ids].values], (y_log - mu) / sd)
        z = m.predict(emb[id_to_emb_idx.loc[pred_ids].values])
        return pred_ids, inverse_target(z, mu, sd)

    rows = []
    for pid in project_ids_all:
        s = splits[pid]
        best_a, best_cv_mae = None, float("inf")
        for a in ALPHAS:
            maes = []
            for fold in s["folds"]:
                ids_v, pred_v = ridge_fit_predict(fold["train"], fold["val"], a)
                maes.append(compute_metrics(id_to_y.loc[ids_v].values, pred_v)[0])
            if np.mean(maes) < best_cv_mae:
                best_a, best_cv_mae = a, float(np.mean(maes))
        ids_t, pred_t = ridge_fit_predict(s["dev"], s["test"], best_a)
        mae, mdae = compute_metrics(id_to_y.loc[ids_t].values, pred_t)
        rows.append({"project_id": pid, "ridge_alpha": best_a, "ridge_cv_mae": best_cv_mae,
                     "ridge_test_mae": mae, "ridge_test_mdae": mdae})
    ridge_df = pd.DataFrame(rows)
    cmp = final_df[["project_id", "project_key", "test_mae", "test_mdae", "baseline_median_mae"]].merge(ridge_df, on="project_id")
    cmp.to_csv(f"{RESULTS_DIR}/cnn_vs_ridge_vs_median.csv", index=False)
    print("Macro test MAE  | CNN: %.4f | ridge: %.4f | median baseline: %.4f" %
          (cmp["test_mae"].mean(), cmp["ridge_test_mae"].mean(), cmp["baseline_median_mae"].mean()))
    print("Macro test MdAE | CNN: %.4f | ridge: %.4f" % (cmp["test_mdae"].mean(), cmp["ridge_test_mdae"].mean()))
    print("Projects where CNN beats ridge (MAE):", int((cmp["test_mae"] < cmp["ridge_test_mae"]).sum()), "/", len(cmp))

Macro test MAE  | CNN: 1.9772 | ridge: 1.9591 | median baseline: 1.9441
Macro test MdAE | CNN: 1.3260 | ridge: 1.3432
Projects where CNN beats ridge (MAE): 11 / 39


In [13]:
# ============================================================
# CELL 10: Figures
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(cv_per_fold["fold"], cv_per_fold["macro_mae"], yerr=cv_per_fold["across_project_sd_mae"].fillna(0), capsize=4)
ax[0].set(xlabel="Temporal validation fold", ylabel="Macro MAE", title="CV macro MAE per fold (error bars: across projects)")
ax[1].bar(cv_per_fold["fold"], cv_per_fold["macro_mdae"], yerr=cv_per_fold["across_project_sd_mdae"].fillna(0), capsize=4)
ax[1].set(xlabel="Temporal validation fold", ylabel="Macro MdAE", title="CV macro MdAE per fold (error bars: across projects)")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_cv_per_fold.png", dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(14, 5))
x, w = np.arange(len(final_df)), 0.2
ax.bar(x - 1.5*w, final_df["test_mae"], w, label="CNN test MAE")
ax.bar(x - 0.5*w, final_df["test_mdae"], w, label="CNN test MdAE")
ax.bar(x + 0.5*w, final_df["baseline_median_mae"], w, label="Median baseline MAE")
if ridge_df is not None:
    ax.bar(x + 1.5*w, final_df[["project_id"]].merge(ridge_df, on="project_id")["ridge_test_mae"], w, label="Ridge test MAE")
ax.set_xticks(x); ax.set_xticklabels(final_df["project_key"], rotation=75, ha="right")
ax.set(xlabel="Project", ylabel="Error (story points)", title="Held-out test error per project")
ax.legend(); plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_test_mae_by_project.png", dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(test_preds_df["y_true"], test_preds_df["y_pred"], s=6, alpha=0.3)
hi = min(SP_MAX, float(test_preds_df["y_true"].max()))
ax.plot([0, hi], [0, hi], "r--", lw=1)
ax.set(xlabel="True story points", ylabel="Predicted", title="Held-out test: predicted vs true")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_scatter_test.png", dpi=150); plt.close()

ae = np.abs(test_preds_df["y_true"].values - test_preds_df["y_pred"].values)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ae, bins=40); ax.set(xlabel="Absolute error", ylabel="Count", title="Held-out test |error| distribution")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_abs_error_hist.png", dpi=150); plt.close()

merged = final_df.merge(cv_per_project[["project_id", "cv_mae_mean", "cv_mae_sd"]], on="project_id", how="left")
fig, ax = plt.subplots(figsize=(14, 5)); x = np.arange(len(merged))
ax.errorbar(x, merged["cv_mae_mean"], yerr=merged["cv_mae_sd"].fillna(0), fmt="o", capsize=3, label="Validation MAE (mean ± SD over folds)")
ax.scatter(x, merged["test_mae"], marker="s", label="Test MAE", zorder=3)
ax.scatter(x, merged["baseline_median_mae"], marker="x", color="gray", label="Median baseline (test)", zorder=2)
ax.set_xticks(x); ax.set_xticklabels(merged["project_key"], rotation=75, ha="right")
ax.set(xlabel="Project", ylabel="MAE (story points)", title="Validation vs held-out test MAE per project")
ax.legend(); plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_val_vs_test_mae.png", dpi=150); plt.close()
print("Figures saved under:", RESULTS_DIR)

Figures saved under: /content/drive/MyDrive/project6_results


In [14]:
# ============================================================
# CELL 11: Manifest of saved results
# ============================================================
total = 0.0
for name in sorted(os.listdir(RESULTS_DIR)):
    p = os.path.join(RESULTS_DIR, name)
    if os.path.isfile(p):
        mb = os.path.getsize(p) / 1e6; total += mb
        print(f"  {name:<42} {mb:>8.3f} MB")
print(f"\n{len(os.listdir(RESULTS_DIR))} files, {total:.2f} MB in {RESULTS_DIR}")
if not IN_COLAB:
    print("Not running in Colab: copy RESULTS_DIR into the repository's results/ folder.")

  best_config_per_project.csv                   0.006 MB
  cnn_vs_ridge_vs_median.csv                    0.005 MB
  config.json                                   0.001 MB
  cv_per_fold_macro_summary.csv                 0.001 MB
  cv_per_project_summary.csv                    0.003 MB
  cv_results_all_configs.csv                    0.096 MB
  cv_results_partial.csv                        0.096 MB
  cv_results_selected_per_project.csv           0.024 MB
  fig_abs_error_hist.png                        0.027 MB
  fig_cv_per_fold.png                           0.045 MB
  fig_scatter_test.png                          0.045 MB
  fig_test_mae_by_project.png                   0.138 MB
  fig_val_vs_test_mae.png                       0.159 MB
  run_meta.json                                 0.004 MB
  summary_report.csv                            0.001 MB
  test_metrics_by_seed.csv                      0.005 MB
  test_metrics_per_project.csv                  0.007 MB
  test_predictions.csv         